#  Sales Forecasting — Complete Walkthrough

Welcome! This notebook walks you through building a sales forecasting project from scratch.
We'll use 3 different models and compare them at the end.

**Run each cell one by one** by pressing `Shift + Enter`.


---
##  Step 1 — Install Libraries

Run this cell once. It installs everything you need.

In [ ]:
# Uncomment and run this cell if you haven't installed the libraries yet
# !pip install pandas numpy matplotlib seaborn statsmodels prophet tensorflow scikit-learn

---
##  Step 2 — Import Libraries

In [ ]:
import warnings
import os
import sys
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('[OK] Libraries imported!')

---
##  Step 3 — Generate Sales Data

We'll create fake-but-realistic sales data for a retail shop.
It has:
- A growth **trend** (business is growing!)
- **Seasonal** patterns (December is huge, January is slow)
- **Promotional** spikes (Black Friday, Christmas)
- Random **noise** (because real life is messy)

In [ ]:
np.random.seed(42)

# Generate 3 years of weekly data
dates  = pd.date_range('2021-01-01', periods=156, freq='W')
n      = len(dates)
base   = 50_000   # average weekly sales in $

# TREND — slow, steady growth
trend  = base * (1.003 ** np.arange(n))

# SEASONALITY — monthly factors
factors = {1:.75,2:.72,3:.80,4:.88,5:.92,6:.95,
           7:1.02,8:1.08,9:.98,10:1.05,11:1.35,12:1.70}
seasonal = np.array([factors[d.month] for d in dates])

# PROMOTIONS — Black Friday and Christmas
promo = np.ones(n)
for i, d in enumerate(dates):
    if d.month == 11 and d.day >= 22: promo[i] = 3.0
    elif d.month == 12 and 19 <= d.day <= 26: promo[i] = 2.5

# NOISE — random variation
noise = np.clip(np.random.normal(1.0, 0.05, n), 0.85, 1.15)

# COMBINE
sales = trend * seasonal * promo * noise

df = pd.DataFrame({'ds': dates, 'y': sales.round(2)})
df.to_csv('../data/sales_data.csv', index=False)

print(f'[OK] Generated {len(df)} weeks of data')
print(f'   Date range: {df.ds.min().date()} → {df.ds.max().date()}')
print(f'   Sales range: ${df.y.min():,.0f} – ${df.y.max():,.0f}/week')
df.head()

---
##  Step 4 — Explore the Data

Before building any model, always **look at your data** first!

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Sales Data Exploration', fontsize=15, fontweight='bold')

# 1. Full time series
ax = axes[0, 0]
ax.plot(df['ds'], df['y'], color='#378ADD', linewidth=1.5)
ax.set_title('Weekly Sales Over Time')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.tick_params(axis='x', rotation=30)

# 2. Monthly average (seasonality)
ax = axes[0, 1]
monthly = df.groupby(df['ds'].dt.month)['y'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
bars = ax.bar(month_names, monthly.values, color='#1D9E75', edgecolor='white')
ax.set_title('Average Sales by Month (Seasonality)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=30)

# 3. Year over year
ax = axes[1, 0]
for year, color in zip([2021, 2022, 2023], ['#378ADD','#1D9E75','#7F77DD']):
    yr = df[df['ds'].dt.year == year]
    if len(yr) > 0:
        ax.plot(yr['ds'].dt.month, yr.groupby(yr['ds'].dt.month)['y'].mean(),
                label=str(year), color=color, linewidth=2, marker='o', markersize=4)
ax.set_title('Year-over-Year Sales Comparison')
ax.set_xticks(range(1,13))
ax.set_xticklabels(month_names)
ax.tick_params(axis='x', rotation=30)
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

# 4. Rolling average (smoothed trend)
ax = axes[1, 1]
ax.plot(df['ds'], df['y'], color='#888780', linewidth=1, alpha=0.5, label='Raw')
ax.plot(df['ds'], df['y'].rolling(4).mean(), color='#378ADD', linewidth=2, label='4-week rolling avg')
ax.plot(df['ds'], df['y'].rolling(12).mean(), color='#E24B4A', linewidth=2, label='12-week rolling avg')
ax.set_title('Rolling Average (Smoothed Trend)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.tick_params(axis='x', rotation=30)
ax.legend()

plt.tight_layout()
plt.savefig('../plots/data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Charts saved to plots/')

---
##  Step 5 — Train / Test Split

We hide the last 20% of data from our models.
Then we use it to check: "How accurate were the predictions?"

This is like a teacher hiding the answer key before giving a test!

In [ ]:
SPLIT = int(len(df) * 0.80)
train = df.iloc[:SPLIT]
test  = df.iloc[SPLIT:]

print(f'Training: {len(train)} weeks  ({train.ds.iloc[0].date()} → {train.ds.iloc[-1].date()})')
print(f'Testing : {len(test)}  weeks  ({test.ds.iloc[0].date()}  → {test.ds.iloc[-1].date()})')

plt.figure(figsize=(14, 5))
plt.plot(train['ds'], train['y'], color='#378ADD', linewidth=1.5, label='Training data (what the model sees)')
plt.plot(test['ds'],  test['y'],  color='#E24B4A', linewidth=1.5, linestyle='--', label='Test data (hidden from model)')
plt.axvline(test['ds'].iloc[0], color='gray', linestyle=':', linewidth=1.5)
plt.title('Train / Test Split')
plt.legend()
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
##  Step 6 — ARIMA Model

ARIMA looks at past sales to predict the next ones.
We use SARIMA which adds seasonal awareness.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

print('Fitting SARIMA... (may take ~30 seconds)')

arima_model = SARIMAX(
    train['y'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 0, 52),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
arima_fit = arima_model.fit(disp=False)

# Predict on test set
arima_test_pred = arima_fit.forecast(steps=len(test))
arima_test_pred.index = test.index

# Forecast 12 weeks into the future
future_dates_a = pd.date_range(
    start=df['ds'].iloc[-1] + pd.Timedelta(weeks=1), periods=12, freq='W')
arima_future   = arima_fit.forecast(steps=len(test)+12).iloc[len(test):]
arima_future.index = future_dates_a

# Metrics
arima_mae  = mean_absolute_error(test['y'], arima_test_pred)
arima_rmse = np.sqrt(mean_squared_error(test['y'], arima_test_pred))
arima_mape = np.mean(np.abs((test['y'].values - arima_test_pred.values) / test['y'].values)) * 100

print(f'[OK] ARIMA done!')
print(f'   MAE : ${arima_mae:,.0f}')
print(f'   RMSE: ${arima_rmse:,.0f}')
print(f'   MAPE: {arima_mape:.2f}%')

plt.figure(figsize=(14, 5))
plt.plot(train['ds'], train['y'], color='#888780', linewidth=1, alpha=0.6, label='Training')
plt.plot(test['ds'],  test['y'],  color='#378ADD', linewidth=2, label='Actual')
plt.plot(test.index, arima_test_pred, color='#E24B4A', linewidth=2, label='ARIMA prediction')
plt.plot(future_dates_a, arima_future, color='#1D9E75', linewidth=2, linestyle='--', label='ARIMA forecast (future)')
plt.title('ARIMA — Forecast vs Actual')
plt.legend()
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../plots/arima_notebook.png', dpi=150, bbox_inches='tight')
plt.show()

---
##  Step 7 — Prophet Model

Prophet is the easiest model. Just give it dates and sales, and it handles everything!

In [ ]:
from prophet import Prophet

# Define holidays Prophet should know about
years = [2021, 2022, 2023, 2024]
holidays = pd.DataFrame({
    'holiday': ['black_friday']*len(years) + ['christmas']*len(years),
    'ds': (
        [pd.Timestamp(f'{y}-11-25') for y in years] +
        [pd.Timestamp(f'{y}-12-25') for y in years]
    ),
    'lower_window': [-1, -1, -1, -1, -7, -7, -7, -7],
    'upper_window': [2, 2, 2, 2, 2, 2, 2, 2],
})

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    holidays=holidays,
    seasonality_mode='multiplicative',
    interval_width=0.95,
)
prophet_model.add_seasonality('monthly', period=30.5, fourier_order=5)
prophet_model.fit(train)  # train only has 'ds' and 'y'

# Make future dataframe and predict
future_df   = prophet_model.make_future_dataframe(periods=len(test)+12, freq='W')
prophet_fc  = prophet_model.predict(future_df)

prophet_test = prophet_fc.iloc[len(train): len(train)+len(test)]
prophet_fut  = prophet_fc.iloc[len(train)+len(test):]

# Metrics
prophet_mae  = mean_absolute_error(test['y'], prophet_test['yhat'])
prophet_rmse = np.sqrt(mean_squared_error(test['y'], prophet_test['yhat']))
prophet_mape = np.mean(np.abs((test['y'].values - prophet_test['yhat'].values) / test['y'].values)) * 100

print(f'[OK] Prophet done!')
print(f'   MAE : ${prophet_mae:,.0f}')
print(f'   RMSE: ${prophet_rmse:,.0f}')
print(f'   MAPE: {prophet_mape:.2f}%')

plt.figure(figsize=(14, 5))
plt.plot(train['ds'], train['y'], color='#888780', linewidth=1, alpha=0.6, label='Training')
plt.plot(test['ds'],  test['y'],  color='#378ADD', linewidth=2, label='Actual')
plt.plot(prophet_test['ds'], prophet_test['yhat'], color='#E24B4A', linewidth=2, label='Prophet prediction')
plt.fill_between(prophet_fut['ds'], prophet_fut['yhat_lower'], prophet_fut['yhat_upper'],
                 color='#EF9F27', alpha=0.25, label='95% confidence')
plt.plot(prophet_fut['ds'], prophet_fut['yhat'], color='#1D9E75', linewidth=2, linestyle='--', label='Prophet forecast')
plt.title('Prophet — Forecast vs Actual')
plt.legend()
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../plots/prophet_notebook.png', dpi=150, bbox_inches='tight')
plt.show()

---
##  Step 8 — LSTM Model

LSTM is the most complex. It's a neural network that has "memory" — it can remember patterns from many weeks ago.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
tf.random.set_seed(42)

SEQ_LEN = 12   # look back 12 weeks

# Scale data to [0, 1]
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[['y']])

# Build sequences
def make_seqs(data, seq):
    X, y = [], []
    for i in range(len(data)-seq):
        X.append(data[i:i+seq])
        y.append(data[i+seq])
    return np.array(X), np.array(y)

train_s = scaled[:SPLIT]
test_s  = scaled[SPLIT-SEQ_LEN:]
X_tr, y_tr = make_seqs(train_s, SEQ_LEN)
X_te, y_te = make_seqs(test_s,  SEQ_LEN)
X_tr = X_tr.reshape(-1, SEQ_LEN, 1)
X_te = X_te.reshape(-1, SEQ_LEN, 1)

# Build and train
lstm_model = Sequential([
    LSTM(64, input_shape=(SEQ_LEN, 1), return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1),
])
lstm_model.compile(optimizer='adam', loss='mse')

print('Training LSTM... (may take 1-2 minutes)')
history = lstm_model.fit(
    X_tr, y_tr, epochs=100, batch_size=16,
    validation_split=0.15,
    callbacks=[EarlyStopping(patience=15, restore_best_weights=True)],
    verbose=0,
)
print(f'[OK] Stopped at epoch {len(history.history["loss"])}')

# Predict on test
lstm_test_pred = scaler.inverse_transform(lstm_model.predict(X_te, verbose=0)).flatten()
lstm_actual    = scaler.inverse_transform(y_te.reshape(-1,1)).flatten()

# Recursive future forecast
last_seq = scaled[-SEQ_LEN:].copy()
lstm_future = []
for _ in range(12):
    nxt = lstm_model.predict(last_seq.reshape(1, SEQ_LEN, 1), verbose=0)[0,0]
    lstm_future.append(nxt)
    last_seq = np.append(last_seq[1:], [[nxt]], axis=0)
lstm_future = scaler.inverse_transform(np.array(lstm_future).reshape(-1,1)).flatten()
future_dates_l = pd.date_range(
    df['ds'].iloc[-1] + pd.Timedelta(weeks=1), periods=12, freq='W')

# Metrics
lstm_mae  = mean_absolute_error(lstm_actual, lstm_test_pred)
lstm_rmse = np.sqrt(mean_squared_error(lstm_actual, lstm_test_pred))
lstm_mape = np.mean(np.abs((lstm_actual - lstm_test_pred) / lstm_actual)) * 100

test_dates_l = df['ds'].values[SPLIT: SPLIT+len(lstm_actual)]

print(f'   MAE : ${lstm_mae:,.0f}')
print(f'   RMSE: ${lstm_rmse:,.0f}')
print(f'   MAPE: {lstm_mape:.2f}%')

plt.figure(figsize=(14, 5))
plt.plot(df['ds'].values[:SPLIT], df['y'].values[:SPLIT], color='#888780', linewidth=1, alpha=0.6, label='Training')
plt.plot(test_dates_l, lstm_actual, color='#378ADD', linewidth=2, label='Actual')
plt.plot(test_dates_l, lstm_test_pred, color='#E24B4A', linewidth=2, label='LSTM prediction')
plt.plot(future_dates_l, lstm_future, color='#1D9E75', linewidth=2, linestyle='--', label='LSTM forecast')
plt.title('LSTM — Forecast vs Actual')
plt.legend()
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../plots/lstm_notebook.png', dpi=150, bbox_inches='tight')
plt.show()

---
##  Step 9 — Compare All Models

In [ ]:
results = {
    'ARIMA':   {'MAE': arima_mae,   'RMSE': arima_rmse,   'MAPE': arima_mape},
    'Prophet': {'MAE': prophet_mae, 'RMSE': prophet_rmse, 'MAPE': prophet_mape},
    'LSTM':    {'MAE': lstm_mae,    'RMSE': lstm_rmse,    'MAPE': lstm_mape},
}

print('='*50)
print(f'{"Model":<12} {"MAE":>10} {"RMSE":>10} {"MAPE":>8}')
print('-'*42)
for name, m in results.items():
    print(f'{name:<12} ${m["MAE"]:>8,.0f} ${m["RMSE"]:>8,.0f} {m["MAPE"]:>7.2f}%')

best = min(results, key=lambda k: results[k]['MAPE'])
print(f'\n Best model: {best}')

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Model Comparison — Lower is Better', fontsize=14, fontweight='bold')
colors = ['#378ADD', '#1D9E75', '#7F77DD']
names = list(results.keys())

for ax, metric, title in zip(axes, ['MAE','RMSE','MAPE'], ['MAE ($)','RMSE ($)','MAPE (%)']):
    vals = [results[n][metric]/1000 if metric != 'MAPE' else results[n][metric] for n in names]
    bars = ax.bar(names, vals, color=colors, width=0.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.02,
                f'{v:.1f}', ha='center', fontsize=10, fontweight='500')
    ax.set_title(title)
    ax.set_ylim(0, max(vals)*1.25)
    for spine in ['top','right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/model_comparison_notebook.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] All done! Check the plots/ folder for all charts.')

---
##  Congratulations!

You just built a complete sales forecasting project with 3 different models.

**What you learned:**
- How to generate and explore time-series data
- How to split data into train/test sets
- How ARIMA, Prophet, and LSTM work
- How to evaluate models with MAE, RMSE, and MAPE
- How to visualize forecasts

**Next steps:**
- Try with your own real data (replace the generator with a CSV import)
- Tune the model parameters to improve accuracy
- Add more features like weather, promotions flags, competitor data
- Deploy the best model as a simple web app with Streamlit!
